In [24]:
import pandas as pd
import xlrd as xl
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 
import numpy as np 

from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

In [8]:
df = pd.read_excel('data/AmesHousing.xlsx')

In [12]:
df_encoded = pd.get_dummies(df, columns=['Neighborhood'], drop_first=True)

In [13]:
feature_cols = ['Overall Qual', 'Gr Liv Area'] + [col for col in df_encoded.columns if 'Neighborhood_' in col]

X = df_encoded[feature_cols]
y = df_encoded['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Vorm van X_train (features voor training):", X_train.shape)
print("Vorm van y_train (target voor training):", y_train.shape)
print("Vorm van X_test (features voor testen):", X_test.shape)
print("Vorm van y_test (target voor testen):", y_test.shape)

Vorm van X_train (features voor training): (2344, 29)
Vorm van y_train (target voor training): (2344,)
Vorm van X_test (features voor testen): (586, 29)
Vorm van y_test (target voor testen): (586,)


In [19]:
model = LinearRegression(fit_intercept=True, n_jobs=-1)

model.fit(X_train, y_train)

print("Het model is succesvol getraind!")

Het model is succesvol getraind!


In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("--- Evaluatiemetrieken Testset ---")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R-kwadraat (R2): {r2:.4f}")

--- Evaluatiemetrieken Testset ---
Mean Absolute Error (MAE): $24,736.74
Root Mean Squared Error (RMSE): $39,052.45
R-kwadraat (R2): 0.8098


In [ ]:
feature_cols_exp = ['Overall Qual', 'Gr Liv Area', 'Year Built', 'Total Bsmt SF'] + \
                   [col for col in df_encoded.columns if 'Neighborhood_' in col]

df_encoded['Total Bsmt SF'] = df_encoded['Total Bsmt SF'].fillna(0)

X_exp = df_encoded[feature_cols_exp]
y_exp = df_encoded['SalePrice']

X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(
    X_exp, y_exp, test_size=0.2, random_state=42
)

In [28]:
model_expA = LinearRegression()

model_expA.fit(X_train_exp, y_train_exp)

y_pred_expA = model_expA.predict(X_test_exp)

print("--- Resultaten Experiment A (Alleen nieuwe features) ---")
print(f"MAE: ${mean_absolute_error(y_test_exp, y_pred_expA):,.2f}")
print(f"R-kwadraat (R2): {r2_score(y_test_exp, y_pred_expA):.4f}")

--- Resultaten Experiment A (Alleen nieuwe features) ---
MAE: $22,532.72
R-kwadraat (R2): 0.8320


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_exp)
X_test_scaled = scaler.transform(X_test_exp)

#Run 1 grotere stappen minder epochs
sgd_baseline = SGDRegressor(max_iter=1000, learning_rate='constant', eta0=0.001, random_state=42)
sgd_baseline.fit(X_train_scaled, y_train_exp)
y_pred_base = sgd_baseline.predict(X_test_scaled)

print("--- RUN 1: SGD Baseline ---")
print(f"Epochs (max_iter): {sgd_baseline.max_iter} | Learning Rate (eta0): {sgd_baseline.eta0}")
print(f"MAE: ${mean_absolute_error(y_test_exp, y_pred_base):,.2f}")
print(f"R-kwadraat (R2): {r2_score(y_test_exp, y_pred_base):.4f}\n")

# Run 2 meer epochs kleinere stappen
sgd_exp = SGDRegressor(max_iter=5000, learning_rate='constant', eta0=0.0001, random_state=42)
sgd_exp.fit(X_train_scaled, y_train_exp)
y_pred_exp = sgd_exp.predict(X_test_scaled)

print("--- RUN 2: SGD Experiment (Meer epochs, kleinere stapjes) ---")
print(f"Epochs (max_iter): {sgd_exp.max_iter} | Learning Rate (eta0): {sgd_exp.eta0}")
print(f"MAE: ${mean_absolute_error(y_test_exp, y_pred_exp):,.2f}")
print(f"R-kwadraat (R2): {r2_score(y_test_exp, y_pred_exp):.4f}")

--- RUN 1: SGD Baseline ---
Epochs (max_iter): 1000 | Learning Rate (eta0): 0.001
MAE: $22,788.16
R-kwadraat (R2): 0.8370

--- RUN 2: SGD Experiment (Meer epochs, kleinere stapjes) ---
Epochs (max_iter): 5000 | Learning Rate (eta0): 0.0001
MAE: $22,560.53
R-kwadraat (R2): 0.8319
